# document_consistency_detector -- round 2 evidence-gate run (Colab)

Round 2: `eval_document_consistency.py` now calibrates its decision
threshold via `best_f1_threshold` (bonafide + train-split fraud only,
applied unchanged to held_out) instead of round 1's hardcoded 0.5 --
round 1 showed a 25% false-positive rate on bonafide documents, plausibly
just an uncalibrated cutoff rather than a detector limitation.

Running this on Colab again (not locally) because local Windows GPU
inference (`paddleocr_env`) hit a real, different failure this time --
`os error 1455`, a Windows virtual-memory commitment-limit error, right
at first inference after a full successful model load. That's a distinct
issue from the DLL-collision class of errors that originally forced round
1 onto Colab, but Colab is still the fastest, most reliable path today
rather than debugging Windows paging-file sizing.

**Before running:** create a small data package locally and upload it
when cell 3 asks for it. From `backend/` (PowerShell):

```powershell
cd D:\GITHUB\red_hat_vs_blue_hat_attack
Compress-Archive -Path "data\generated\document_bonafide","data\generated\document_attacks" -DestinationPath "document_fraud_data.zip" -Force
```

That zips the 40 bonafide documents + all generated document_fraud attack
cases (train + held_out) and their tampered images -- everything
`eval_document_consistency.py` reads from disk. No credentials, no code
needed in the zip -- the code is written directly by cell 2 below, same
approach as the GNN notebook.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine) before
running cell 1.


## 1. Install dependencies

Same package list as `requirements-paddleocr-gpu.txt` -- deliberately NOT installing a separate `opencv-python` (see that file's own comment: paddlex pins `opencv-contrib-python` itself, and installing both corrupts the cv2 native module).

In [ ]:
# paddlepaddle-gpu is NOT pulled in automatically by paddleocr[doc-parser] --
# confirmed by a real run (EVAL FAILED: Engine 'paddle_static' is unavailable
# because dependency 'paddlepaddle' is not installed). Installing it
# explicitly from PaddlePaddle's own package index, matching the exact
# working local setup already documented in requirements-paddleocr-gpu.txt
# (cu126 index, requires GPU driver >=550.54.14 -- Colab's T4 driver is
# comfortably newer). Installed BEFORE paddleocr so its own dependency
# resolution finds paddlepaddle already satisfied rather than pulling in a
# conflicting CPU-only build.
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q numpy scikit-learn supabase python-dotenv "paddleocr[doc-parser]"

# The paddlepaddle-gpu install above silently downgraded three CUDA runtime
# packages (nvidia-cudnn-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12) to
# versions older than what Colab's preinstalled torch was compiled against
# -- confirmed by a real run: pip's own resolver warning named the exact
# versions torch requires, and a real crash later (undefined symbol:
# ncclCommShrink from libtorch_cuda.so) confirmed something in the
# paddleocr[doc-parser] import chain opportunistically imports torch and
# needs it working. Restoring those three packages to the exact versions
# torch's own pip warning named -- newer runtime libraries satisfying an
# older/looser paddle requirement is the normal compatible direction (the
# crash happened in the other direction: torch's binary calling a symbol
# that only exists in the newer nccl it was actually built against).
!pip install -q --force-reinstall "nvidia-cudnn-cu12==9.19.0.56" "nvidia-cusparselt-cu12==0.7.1" "nvidia-nccl-cu12==2.28.9"

import paddle
print("paddle:", paddle.__version__, "| CUDA compiled:", paddle.device.is_compiled_with_cuda(),
      "| GPU count:", paddle.device.cuda.device_count() if paddle.device.is_compiled_with_cuda() else 0)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Write the backend source files

Same files already committed to the real repo (round-2 threshold-calibration fix included) -- embedded directly here so this notebook is self-contained, the same pattern `train_gnn_mule_network.ipynb` uses. No repo clone needed (`data/generated/` is gitignored anyway, so a clone wouldn't bring the images regardless).

In [ ]:
import os, pathlib

_FILES = {}
_FILES['backend/__init__.py'] = ''
_FILES['backend/db/__init__.py'] = ''
_FILES['backend/db/supabase_client.py'] = '"""\nThin Supabase client factory for the backend. Two entry points, deliberately\nkept separate so a script can\'t accidentally write with a read-only key or\nleak the service-role key into anything client-facing:\n\n- get_service_client(): service-role key, bypasses RLS -- used by every\n  backend script that WRITES (backfill, training scripts recording to\n  model_registry, the evaluation harness recording runs/results).\n- get_anon_client(): anon/publishable key, subject to the "public read"\n  RLS policies in migrations/002_rls_policies.sql -- used anywhere a\n  read-only client is enough (mirrors what the frontend does).\n\nBoth read credentials from environment variables (see .env.example at the\nrepo root) via python-dotenv -- never hardcoded, never committed.\n"""\n\nimport os\nfrom functools import lru_cache\nfrom pathlib import Path\n\nfrom dotenv import load_dotenv\nfrom supabase import Client, create_client\n\n# Real credentials live in backend/.env (SUPABASE_URL / SUPABASE_ANON_KEY /\n# SUPABASE_SERVICE_ROLE_KEY are already populated there). Load it explicitly\n# by path so this works regardless of the caller\'s cwd -- a bare load_dotenv()\n# only checks cwd and would silently miss it when a script is run from the\n# repo root. Falls back to a repo-root .env if one exists (won\'t override\n# values already loaded from backend/.env, load_dotenv defaults to non-clobber).\n_BACKEND_DIR = Path(__file__).resolve().parents[1]\nload_dotenv(_BACKEND_DIR / ".env")\nload_dotenv()\n\nSUPABASE_URL = os.environ.get("SUPABASE_URL")\nSUPABASE_ANON_KEY = os.environ.get("SUPABASE_ANON_KEY")\nSUPABASE_SERVICE_ROLE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")\n\n\ndef _require(value: str | None, name: str) -> str:\n    if not value:\n        raise RuntimeError(\n            f"{name} is not set. Copy .env.example to .env at the repo root and fill it in "\n            f"(SUPABASE_URL / SUPABASE_ANON_KEY / SUPABASE_SERVICE_ROLE_KEY -- "\n            f"Project Settings -> API in the Supabase dashboard)."\n        )\n    return value\n\n\n@lru_cache(maxsize=1)\ndef get_service_client() -> Client:\n    """Service-role client -- bypasses RLS. Writes only. Never expose this key\n    to the frontend or commit it anywhere."""\n    url = _require(SUPABASE_URL, "SUPABASE_URL")\n    key = _require(SUPABASE_SERVICE_ROLE_KEY, "SUPABASE_SERVICE_ROLE_KEY")\n    return create_client(url, key)\n\n\n@lru_cache(maxsize=1)\ndef get_anon_client() -> Client:\n    """Anon/publishable-key client -- subject to the public-read RLS policies.\n    Use this for anything that should behave the same way the frontend does."""\n    url = _require(SUPABASE_URL, "SUPABASE_URL")\n    key = _require(SUPABASE_ANON_KEY, "SUPABASE_ANON_KEY")\n    return create_client(url, key)\n'
_FILES['backend/evaluation/__init__.py'] = ''
_FILES['backend/evaluation/metrics.py'] = '"""\nShared metrics computation -- precision/recall/F1/ROC-AUC/PR-AUC/false\npositive rate, used by every training script (Stage 5) and, later, the\nadversarial evaluation harness (Stage 7). One place so "how we compute a\nmetric" can\'t drift between models (docs/TECHNICAL_SPEC.md Section 8 step\n4 names this exact metric set).\n"""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score, confusion_matrix, f1_score,\n    precision_recall_curve, precision_score, recall_score, roc_auc_score,\n)\n\n\ndef compute_binary_metrics(y_true, y_score, threshold: float = 0.5) -> dict:\n    y_true = np.asarray(y_true)\n    y_score = np.asarray(y_score)\n    y_pred = (y_score >= threshold).astype(int)\n\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n    fpr = fp / (fp + tn) if (fp + tn) else 0.0\n    multi_class_present = len(np.unique(y_true)) > 1\n\n    return {\n        "threshold": float(threshold),\n        "precision": float(precision_score(y_true, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true, y_pred, zero_division=0)),\n        "f1": float(f1_score(y_true, y_pred, zero_division=0)),\n        "roc_auc": float(roc_auc_score(y_true, y_score)) if multi_class_present else None,\n        "pr_auc": float(average_precision_score(y_true, y_score)) if multi_class_present else None,\n        "false_positive_rate": float(fpr),\n        "n_samples": int(len(y_true)),\n        "n_positive": int(y_true.sum()),\n        "true_positives": int(tp), "false_positives": int(fp),\n        "true_negatives": int(tn), "false_negatives": int(fn),\n    }\n\n\ndef best_f1_threshold(y_true, y_score) -> float:\n    """Threshold that maximizes F1 on (y_true, y_score) -- used to pick an\n    operating point for the confusion-matrix-based metrics above. ROC-AUC\n    and PR-AUC themselves don\'t need a threshold; this is only for\n    precision/recall/F1 at a single reported cutoff.\n    """\n    precision, recall, thresholds = precision_recall_curve(y_true, y_score)\n    if not len(thresholds):\n        return 0.5\n    f1 = 2 * precision * recall / (precision + recall + 1e-12)\n    return float(thresholds[np.argmax(f1[:-1])])\n\n\ndef record_result(results_path, model_name: str, metrics: dict, extra: dict | None = None) -> None:\n    """Upsert one model\'s metrics into a shared JSON results file (keyed by\n    model_name, so re-running a training script updates its own entry\n    without disturbing the others).\n    """\n    results_path = Path(results_path)\n    data = json.loads(results_path.read_text()) if results_path.exists() else {}\n    entry = {"metrics": metrics}\n    if extra:\n        entry.update(extra)\n    data[model_name] = entry\n    results_path.parent.mkdir(parents=True, exist_ok=True)\n    results_path.write_text(json.dumps(data, indent=2))\n'
_FILES['backend/evaluation/supabase_results.py'] = '"""\nTask #32 -- shared helper so every eval_*.py script persists real per-case\nresults into Supabase\'s `evaluation_runs` / `evaluation_results` tables\n(001_core_schema.sql), not just an aggregate metrics.json entry. This is\nwhat the evidence-viewer frontend page reads: one real row per scored\ncase, with the detector\'s own score and reasoning trace, matched against\nthe case\'s ground truth only in the `actual_label` column here -- exactly\nPrinciple 13\'s boundary, enforced by construction: this module runs AFTER\na detector\'s score() has already been called, never before.\n\nfused_risk_score is NOT true multi-signal fusion (Section 6) -- that layer\ndoesn\'t exist as code yet (#33-36). It\'s the single detector\'s own score\nscaled to the 0-100 band so the decision-band table in Section 6 can be\napplied consistently today; every row\'s evidence/model_signals makes clear\nonly one signal contributed. Re-running an eval script re-creates a fresh\nevaluation_runs row each time (not upserted) -- each run is its own\nhistorical record, matching evaluation_runs\' own append-only shape.\n"""\n\nfrom datetime import datetime, timezone\n\nfrom defend.fusion import decision_for as _decision_for  # noqa: E402  -- Section 6 decision bands, single source (defend/fusion.py)\n\n\ndef record_run_and_results(client, run_type: str, model_name: str, cases: list, batch_size: int = 200) -> str:\n    """cases: list of dicts with keys case_id, score (0-1), threshold,\n    is_fraud (bool), evidence (list[str]). Creates one evaluation_runs row\n    and one evaluation_results row per case. Returns the new run_id."""\n    now = datetime.now(timezone.utc).isoformat()\n    run_resp = client.table("evaluation_runs").insert({\n        "run_type": run_type,\n        "config": {"model": model_name, "n_cases": len(cases)},\n        "status": "completed",\n        "started_at": now,\n        "finished_at": now,\n    }).execute()\n    run_id = run_resp.data[0]["id"]\n\n    rows = []\n    for c in cases:\n        fused = round(c["score"] * 100, 2)\n        predicted_fraud = c["score"] >= c["threshold"]\n        rows.append({\n            "run_id": run_id,\n            "case_id": c["case_id"],\n            "model_signals": [{"model": model_name, "score": c["score"]}],\n            "fused_risk_score": fused,\n            "decision": _decision_for(fused),\n            "detected": bool(predicted_fraud == c["is_fraud"]),\n            "actual_label": "fraud" if c["is_fraud"] else "legit",\n            "evidence": c.get("evidence", []),\n        })\n\n    for i in range(0, len(rows), batch_size):\n        client.table("evaluation_results").insert(rows[i:i + batch_size]).execute()\n\n    return run_id\n'
_FILES['backend/evaluation/eval_document_consistency.py'] = '"""\nPrinciple 11 evidence gate for the document_consistency detector\n(defend/pretrained/document_consistency_detector.py): scores our own\nbonafide (fully consistent, document_gen.generate_bonafide_documents) and\nfraud (tampered invoices, generate_document_attacks.py) documents, records\nreal precision/recall/ROC-AUC/PR-AUC -- whatever they turn out to be -- to\nbackend/defend/models/metrics.json and docs/EVALUATION_RESULTS.md.\nStructurally identical to eval_voice_spoof.py; see that file for the\ngeneral pattern this follows.\n\nReports the held_out split separately from train -- held_out\'s\n"amount + beneficiary + QR tampered together" combination is the\nmulti-field simultaneous case (Section 4a); train only ever tampers one\nfield at a time, so a recall gap between splits here is itself a real\nfinding, same as for voice_spoof and the tabular families.\n\nCaveat recorded, not hidden: the field-extraction regexes in\ndocument_consistency_detector.py are keyed to this project\'s own\ndocument_gen.py label format -- this evaluates "can PaddleOCR + our own\nconsistency logic catch tampering on documents we render," not general\nreal-world invoice fraud detection.\n\nNOT executable in the cloud sandbox this was authored in -- depends on\ndocument_consistency_detector.py (paddleocr/paddlepaddle/opencv) and real\ngenerated images from generate_document_attacks.py.\n\nUsage:\n    python backend/evaluation/eval_document_consistency.py\n"""\n\nimport json\nimport sys\nfrom pathlib import Path\n\nBACKEND_DIR = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(BACKEND_DIR))\n\nimport time  # noqa: E402\n\nimport numpy as np  # noqa: E402\n\nfrom db.supabase_client import get_service_client  # noqa: E402\nfrom defend.pretrained.document_consistency_detector import DocumentConsistencyDetector  # noqa: E402\nfrom evaluation.metrics import best_f1_threshold, compute_binary_metrics, record_result  # noqa: E402\nfrom evaluation.supabase_results import record_run_and_results  # noqa: E402\n\nREPO_ROOT = BACKEND_DIR.parent\nBONAFIDE_DIR = REPO_ROOT / "data" / "generated" / "document_bonafide"\nDOCUMENT_ATTACKS_DIR = REPO_ROOT / "data" / "generated" / "document_attacks"\nMODELS_DIR = BACKEND_DIR / "defend" / "models"\nRESULTS_JSON = MODELS_DIR / "metrics.json"\nRESULTS_MD = BACKEND_DIR.parent / "docs" / "EVALUATION_RESULTS.md"\n\n\ndef _append_results_md(overall: dict, per_split: dict, n_bonafide: int, threshold: float) -> None:\n    RESULTS_MD.parent.mkdir(parents=True, exist_ok=True)\n    if not RESULTS_MD.exists():\n        RESULTS_MD.write_text(\n            "# Evaluation Results\\n\\n"\n            "Recorded automatically by each training/evaluation script -- do not hand-edit\\n"\n            "numbers here, re-run the script instead.\\n"\n        )\n    lines = [\n        "\\n## document_consistency_detector (PaddleOCR-VL + QR cross-check) "\n        "-- Principle 11 evidence-gate run\\n",\n        f"- Decision threshold: {threshold:.4f} (best_f1_threshold on bonafide + train-split fraud "\n        f"only, then applied unchanged to held_out -- same calibrate-on-train/apply-to-held_out "\n        f"pattern as run_adversarial_eval.py\'s frozen tabular thresholds, not re-picked on held_out)",\n        f"- Precision: {overall[\'precision\']:.4f}",\n        f"- Recall: {overall[\'recall\']:.4f}",\n        f"- F1: {overall[\'f1\']:.4f}",\n        f"- ROC-AUC: {overall[\'roc_auc\']:.4f}",\n        f"- PR-AUC: {overall[\'pr_auc\']:.4f}",\n        f"- False positive rate (bonafide flagged as tampered): {overall[\'false_positive_rate\']:.4%}",\n        f"- n_bonafide={n_bonafide} (self-generated, see script docstring), "\n        f"n_fraud={overall[\'n_samples\'] - n_bonafide}",\n    ]\n    for split, m in per_split.items():\n        lines.append(f"- {split} split recall: {m[\'recall\']:.4f} (n={m[\'n_positive\']})")\n    lines.append("")\n    with open(RESULTS_MD, "a") as f:\n        f.write("\\n".join(lines))\n\n\ndef _score_with_progress(detector, paths: list, label: str) -> tuple:\n    """score_batch() is a silent list comprehension -- on CPU, a single\n    PaddleOCR-VL forward pass can take long enough that a genuinely-working\n    run and a genuinely-hung one look identical on screen. Print per-image\n    timing so that ambiguity never happens again. Uses score_with_evidence()\n    (Task #32) so the evidence viewer gets real per-field mismatch detail,\n    not just the number -- score() itself is unchanged, still Principle-13\n    compliant (file path only)."""\n    scores = []\n    evidences = []\n    for i, path in enumerate(paths, 1):\n        t0 = time.monotonic()\n        score, evidence = detector.score_with_evidence(path)\n        dt = time.monotonic() - t0\n        scores.append(score)\n        evidences.append(evidence)\n        print(f"  [{label}] {i}/{len(paths)}  ({dt:.1f}s this image)", flush=True)\n    return np.array(scores, dtype="float64"), evidences\n\n\ndef main() -> None:\n    if not BONAFIDE_DIR.exists() or not any(BONAFIDE_DIR.glob("*.png")):\n        raise FileNotFoundError(\n            f"No bonafide documents under {BONAFIDE_DIR}. Run generate/generate_document_attacks.py "\n            f"first (it generates these as a side effect)."\n        )\n    case_paths = sorted(DOCUMENT_ATTACKS_DIR.glob("*/*.json"))\n    if not case_paths:\n        raise FileNotFoundError(\n            f"No generated document_fraud cases under {DOCUMENT_ATTACKS_DIR}. Run "\n            f"generate/generate_document_attacks.py first."\n        )\n\n    detector = DocumentConsistencyDetector()\n\n    bonafide_paths = sorted(BONAFIDE_DIR.glob("*.png"))\n    print(f"Scoring {len(bonafide_paths)} bonafide documents...")\n    bonafide_scores, bonafide_evidence = _score_with_progress(detector, bonafide_paths, "bonafide")\n\n    cases = [json.loads(p.read_text()) for p in case_paths]\n    image_paths = [REPO_ROOT / c["image_path"].replace("\\\\", "/") for c in cases]\n    print(f"Scoring {len(image_paths)} generated (tampered) documents...")\n    fraud_scores, fraud_evidence = _score_with_progress(detector, image_paths, "fraud")\n\n    # Calibrate the decision threshold on bonafide + TRAIN-split fraud cases only\n    # (best_f1_threshold -- same helper run_adversarial_eval.py/eval_fusion.py use\n    # for every other detector), then apply that ONE fixed threshold everywhere\n    # below including held_out. Previously hardcoded 0.5 -- an unexamined default,\n    # same real gap as eval_voice_spoof.py\'s pre-fix version.\n    train_idx = [i for i, c in enumerate(cases) if c["split_portion"] == "train"]\n    train_fraud_scores = fraud_scores[train_idx]\n    calib_true = np.concatenate([np.zeros(len(bonafide_scores)), np.ones(len(train_fraud_scores))])\n    calib_score = np.concatenate([bonafide_scores, train_fraud_scores])\n    threshold = best_f1_threshold(calib_true, calib_score)\n    print(f"Calibrated decision threshold (best-F1 on bonafide + train-split fraud): {threshold:.4f}")\n\n    y_true = np.concatenate([np.zeros(len(bonafide_scores)), np.ones(len(fraud_scores))])\n    y_score = np.concatenate([bonafide_scores, fraud_scores])\n    overall = compute_binary_metrics(y_true, y_score, threshold=threshold)\n    print(f"Overall metrics:\\n{json.dumps(overall, indent=2)}")\n\n    per_split = {}\n    for split in ("train", "held_out"):\n        idx = [i for i, c in enumerate(cases) if c["split_portion"] == split]\n        if not idx:\n            continue\n        split_scores = fraud_scores[idx]\n        split_y_true = np.ones(len(split_scores))\n        combined_true = np.concatenate([np.zeros(len(bonafide_scores)), split_y_true])\n        combined_score = np.concatenate([bonafide_scores, split_scores])\n        per_split[split] = compute_binary_metrics(combined_true, combined_score, threshold=threshold)\n        print(f"  {split} (n={len(idx)}): recall={per_split[split][\'recall\']:.4f}")\n\n    record_result(\n        RESULTS_JSON, "document_consistency_detector", overall,\n        extra={\n            "decision_threshold": threshold,\n            "n_bonafide": len(bonafide_scores),\n            "n_fraud": len(fraud_scores),\n            "held_out_recall": per_split.get("held_out", {}).get("recall"),\n            "train_recall": per_split.get("train", {}).get("recall"),\n            "note": ("pretrained OCR-VL + our own QR cross-check logic, no training -- "\n                     "evidence-gate run per Principle 11. Threshold calibrated via "\n                     "best_f1_threshold on bonafide + train-split fraud only (round 2), then "\n                     "applied unchanged to held_out -- round 1 used an uncalibrated 0.5 default."),\n        },\n    )\n    _append_results_md(overall, per_split, len(bonafide_scores), threshold)\n    print(f"Recorded results to {RESULTS_JSON} and {RESULTS_MD}")\n\n    # Task #32: per-case results into Supabase for the evidence viewer.\n    # Best-effort, non-fatal -- see eval_phishing_classifier.py for why.\n    try:\n        client = get_service_client()\n        bonafide_records = [\n            {"case_id": bonafide_paths[i].stem, "score": float(bonafide_scores[i]),\n             "threshold": threshold, "is_fraud": False, "evidence": bonafide_evidence[i]}\n            for i in range(len(bonafide_paths))\n        ]\n        for split in ("train", "held_out"):\n            idx = [i for i, c in enumerate(cases) if c["split_portion"] == split]\n            if not idx:\n                continue\n            split_records = [\n                {"case_id": cases[i]["case_id"], "score": float(fraud_scores[i]),\n                 "threshold": threshold, "is_fraud": True, "evidence": fraud_evidence[i]}\n                for i in idx\n            ]\n            run_type = "adversarial_train_eval" if split == "train" else "adversarial_held_out"\n            run_id = record_run_and_results(\n                client, run_type=run_type, model_name="document_consistency_detector",\n                cases=bonafide_records + split_records,\n            )\n            print(f"  Supabase: evaluation_run {run_id} ({run_type}, "\n                  f"{len(bonafide_records) + len(split_records)} per-case results)")\n    except Exception as exc:\n        print(f"  Supabase per-case persistence skipped (non-fatal): {exc}", file=sys.stderr)\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except Exception as exc:\n        print(f"\\nEVAL FAILED: {exc}", file=sys.stderr)\n        sys.exit(1)\n'
_FILES['backend/defend/__init__.py'] = ''
_FILES['backend/defend/fusion.py'] = '"""\nSection 6 (docs/TECHNICAL_SPEC.md) -- the real fusion layer. Until\n2026-08-30 this didn\'t exist as code: evaluation/supabase_results.py\'s own\ndocstring says fused_risk_score was "the single detector\'s own score\nscaled to the 0-100 band," not true multi-signal fusion. This module is\nwhat makes that real for the four tabular attack families\n(transaction_fraud, account_takeover, synthetic_identity, mule_network),\nwhere XGBoost, LightGBM, and Autoencoder all score the same transaction.\n\nFUSION_WEIGHTS are read from backend/defend/models/metrics.json\'s real\nStage-5 validation ROC-AUC per model -- not hand-picked, not the spec\'s\n"equal-ish" placeholder forever. ROC-AUC (not the threshold-dependent\nrecall/precision numbers) is the right thing to weight by here: it\nreflects each model\'s ranking quality independent of any one operating\nthreshold, which is what a weighted-average fusion actually consumes.\nComputed once at import time from a real file, auditable by reading that\nfile, not asserted.\n\nDecision bands (0-30 approve / 31-60 review / 61-80 challenge / 81-100\nblock) live here now as the single canonical source -- previously\nduplicated in evaluation/supabase_results.py, which now imports from here.\n\nBehavioral corroboration (Section 6 / Principle 14, Customer Universe\'s\nbehavior_baseline -- see docs/TECHNICAL_SPEC.md Section 4b-i) is\nimplemented as a pure function below, unit-testable and ready to use, but\nNOT yet exercised against real evaluation data: attack_cases.customer_id\nlinkage is still None for every generated case\n(backend/db/backfill_attack_cases.py\'s own comment: "identity-family\nlinkage arrives in Phase 2.5"). Stated here plainly rather than silently\nvalidated against nothing -- this piece is built, not yet evidence-gated\nper Principle 11.\n"""\n\nimport json\nfrom pathlib import Path\n\nMODELS_DIR = Path(__file__).resolve().parent / "models"\nMETRICS_PATH = MODELS_DIR / "metrics.json"\n\nTABULAR_MODELS = ("xgboost", "lightgbm", "autoencoder")\n\n# Section 6 decision bands -- canonical source (evaluation/supabase_results.py imports this).\nDECISION_BANDS = ((30, "approve"), (60, "review"), (80, "challenge"), (100, "block"))\n\n\ndef decision_for(fused_score_0_100: float) -> str:\n    for ceiling, decision in DECISION_BANDS:\n        if fused_score_0_100 <= ceiling:\n            return decision\n    return "block"\n\n\ndef compute_fusion_weights(metrics_path: Path = METRICS_PATH) -> dict:\n    """Weight per tabular model = its Stage-5 validation ROC-AUC, normalized\n    to sum to 1. Read fresh from metrics.json every call (not cached at\n    import time) so a re-run of train_xgboost.py/train_lightgbm.py/\n    train_autoencoder.py is automatically reflected -- this file is the\n    single source of truth for what "how much do we trust this model"\n    means, and it should never silently go stale against a retrained model."""\n    data = json.loads(Path(metrics_path).read_text())\n    raw = {}\n    for m in TABULAR_MODELS:\n        if m not in data:\n            raise KeyError(f"metrics.json has no entry for \'{m}\' -- run its train_*.py script first.")\n        roc_auc = data[m]["metrics"]["roc_auc"]\n        if roc_auc is None:\n            raise ValueError(f"metrics.json\'s \'{m}\' entry has no roc_auc recorded.")\n        raw[m] = float(roc_auc)\n    total = sum(raw.values())\n    return {m: v / total for m, v in raw.items()}\n\n\ndef fuse_tabular_scores(scores: dict, weights: dict = None) -> float:\n    """scores: {model_name: raw probability in [0,1]} -- need not include\n    every TABULAR_MODELS key; missing signals are excluded and remaining\n    weights renormalized (a live request might not always run all three,\n    e.g. if Autoencoder inference is unavailable). Returns a 0-100 fused\n    risk score, a weighted average -- not a re-derived threshold or a\n    max/min rule, per Section 6\'s "weighted-ish" starting point, now\n    grounded in real per-model ROC-AUC rather than a guess."""\n    if not scores:\n        raise ValueError("fuse_tabular_scores called with no signals")\n    weights = weights or compute_fusion_weights()\n    active = {m: w for m, w in weights.items() if m in scores}\n    if not active:\n        raise ValueError(f"None of {list(scores)} are known tabular models ({TABULAR_MODELS})")\n    norm = sum(active.values())\n    fused = sum(scores[m] * (w / norm) for m, w in active.items())\n    return round(fused * 100, 2)\n\n\ndef behavioral_adjustment(fused_score_0_100: float, transaction: dict, behavior_baseline: dict):\n    """Section 6 / Principle 14: corroborate or discount a fused score using\n    the customer\'s own behavior_baseline (generate/synthetic_customers.py\'s\n    _generate_behavior_baseline() -- normal_amount_range, occasional_*\n    ranges for country/channel/login_hour, etc). Returns (adjusted_score,\n    reason_string).\n\n    Deliberately conservative and explainable, not a learned model: a\n    transaction inside the customer\'s own normal_* ranges is DISCOUNTED\n    (a borderline detector signal is more likely a false positive when it\n    matches this specific customer\'s known-normal behavior); a\n    transaction outside BOTH normal_* and occasional_* ranges on 2+\n    dimensions is CORROBORATED (boosted) -- otherwise the score passes\n    through unchanged. NOT yet run against real evaluation data -- see\n    this module\'s docstring; ready for use once attack_cases.customer_id\n    linkage exists (Phase 2.5)."""\n    if not behavior_baseline:\n        return fused_score_0_100, "no behavior_baseline available for this customer -- score unadjusted"\n\n    outside_count = 0\n    inside_normal_count = 0\n    checked = 0\n\n    amount = transaction.get("amount")\n    normal_amt = behavior_baseline.get("normal_amount_range")\n    occ_amt = behavior_baseline.get("occasional_amount_range")\n    if amount is not None and normal_amt and occ_amt:\n        checked += 1\n        if normal_amt[0] <= amount <= normal_amt[1]:\n            inside_normal_count += 1\n        elif not (occ_amt[0] <= amount <= occ_amt[1]):\n            outside_count += 1\n\n    country = transaction.get("country")\n    normal_countries = behavior_baseline.get("normal_countries")\n    occ_countries = behavior_baseline.get("occasional_countries")\n    if country is not None and normal_countries is not None:\n        checked += 1\n        if country in normal_countries:\n            inside_normal_count += 1\n        elif occ_countries is None or country not in occ_countries:\n            outside_count += 1\n\n    channel = transaction.get("channel")\n    normal_channels = behavior_baseline.get("normal_channels")\n    occ_channels = behavior_baseline.get("occasional_channels")\n    if channel is not None and normal_channels is not None:\n        checked += 1\n        if channel in normal_channels:\n            inside_normal_count += 1\n        elif occ_channels is None or channel not in occ_channels:\n            outside_count += 1\n\n    if checked == 0:\n        return fused_score_0_100, "transaction missing all comparable fields -- score unadjusted"\n\n    if outside_count >= 2:\n        adjusted = min(100.0, fused_score_0_100 * 1.15)\n        return round(adjusted, 2), f"corroborated: {outside_count}/{checked} dimensions outside both normal and occasional ranges"\n    if inside_normal_count == checked:\n        adjusted = fused_score_0_100 * 0.7\n        return round(adjusted, 2), f"discounted: all {checked} comparable dimensions match this customer\'s normal behavior"\n    return fused_score_0_100, f"no strong corroboration or discount ({inside_normal_count} normal, {outside_count} outside, {checked} checked)"\n'
_FILES['backend/defend/pretrained/__init__.py'] = ''
_FILES['backend/defend/pretrained/document_consistency_detector.py'] = '"""\nThin wrapper around pretrained OCR (PaddleOCR-VL) for document_fraud\n(Section 4a) detection -- Section 5\'s "OCR / document consistency\n(PaddleOCR)... No -- pretrained inference" row, same Principle 6 rationale\nas voice_spoof_detector.py.\n\nUnlike the voice detector, there is no single pretrained "is this invoice\ntampered" model to load -- PaddleOCR-VL does document parsing (layout +\ntext), not invoice-field extraction or consistency-checking. The\nconsistency check itself (does the PRINTED text match the QR-decoded\npayload) is our own logic layered on top of its parsed output, keyed to\nthe specific label format generate/artifact_generators/document_gen.py\nrenders ("Invoice #:", "Payable to:", "GRAND TOTAL:", "A/C No.:") -- this\nis legitimate because we control that format ourselves; it would NOT\ngeneralize to arbitrary real-world invoice layouts without a lot more\nwork, and that limitation is recorded here rather than hidden.\n\n2026-08-30: document_gen.py\'s template was rewritten from a minimal\nplaceholder to a realistic multi-section tax invoice, and a fourth\ntamperable field (bank_account) was added -- the label regexes below were\nupdated in lockstep. "Amount" -> "GRAND TOTAL" specifically (not "Total")\nbecause the new template also prints "Subtotal:", and a bare `Total` regex\nwould false-match inside "Subtotal" (the substring "total" literally\nappears in "Subtotal"); anchoring on "GRAND TOTAL" avoids that. Similarly\n"Pay to:" -> "Payable to:" to match the new Payment Details section.\n\nUses PaddleOCR-VL (PaddlePaddle/PaddleOCR-VL-1.6, 0.9B params) rather than\nplain PP-OCR text recognition -- 2026-08-30 decision: stronger document-\nstructure understanding for effectively the same install/dependency\nfootprint, since it still runs on paddlepaddle, not a separate torch-based\nVLM stack. A genuinely separate VLM reasoning layer was considered and\ndeliberately deferred -- see docs/FUTURE_INTEGRATIONS.md, item 1.\n\nPaddleOCR-VL API used (github.com/PaddlePaddle/PaddleOCR docs,\nversion3.x/pipeline_usage/PaddleOCR-VL.en.md, verified 2026-08-30):\n    from paddleocr import PaddleOCRVL\n    pipeline = PaddleOCRVL()\n    output = pipeline.predict(image_path)\n    for res in output:\n        res.markdown["markdown_texts"]  # plain-text/markdown page content\nRequires the `paddleocr[doc-parser]` install extra, not plain `paddleocr`.\n\nQR decoding uses OpenCV\'s built-in cv2.QRCodeDetector -- already a\ntransitive dependency of paddleocr, so no separate QR-reading library\nneeded.\n\nIdentity-consistency-vs-customer-profile (does the beneficiary match this\ncase\'s customer_id\'s trusted_beneficiaries?) is deliberately NOT built\ninto this class -- it needs no model and no evidence gate (it\'s a plain\ndict lookup against generate/synthetic_customers.py\'s roster), so it\nbelongs in the API layer (Task #36) that already has the case\'s\ncustomer_id in hand, not duplicated here.\n"""\n\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\n\n# Keyed to document_gen.py\'s exact rendered label text -- change one, change both.\n# Anchored to our own generated format (INV-########) directly, not the\n# "Invoice #:" label -- the new template also prints "TAX INVOICE" as a\n# header line ahead of the real "Invoice #:" line, and a label-based regex\n# risks latching onto that occurrence instead. We control the exact format\n# we generate, so matching it directly sidesteps the ambiguity entirely.\n_INVOICE_NUMBER_RE = re.compile(r"(INV-\\d+)", re.IGNORECASE)\n_BENEFICIARY_RE = re.compile(r"Payable\\s*to:?\\s*(.+)", re.IGNORECASE)\n_AMOUNT_RE = re.compile(r"GRAND\\s*TOTAL:?\\s*\\$?\\s*([\\d,]+\\.?\\d*)", re.IGNORECASE)\n_BANK_ACCOUNT_RE = re.compile(r"A/C\\s*No\\.?:?\\s*([A-Za-z0-9\\-]+)", re.IGNORECASE)\n\n\nclass DocumentConsistencyDetector:\n    """Lazy-loads PaddleOCR-VL on first use (not at import time) so\n    importing this module doesn\'t require paddleocr/paddlepaddle unless\n    the detector is actually used."""\n\n    def __init__(self):\n        self._ocr = None\n\n    def _ensure_loaded(self) -> None:\n        if self._ocr is not None:\n            return\n        from paddleocr import PaddleOCRVL\n\n        # use_queues=False (2026-08-30): PaddleOCR-VL\'s downloaded pipeline\n        # config defaults use_queues=True, routing every predict() call\n        # through a threaded CV/VLM producer-consumer pipeline (separate\n        # worker threads handing off through a queue.Queue) -- confirmed by\n        # reading paddlex/inference/pipelines/paddleocr_vl/pipeline.py\n        # directly. That design deadlocks on a single-image predict() call:\n        # the main thread blocks forever on queue_vlm.get(timeout=0.5)\n        # because the VLM worker thread never receives its own terminating\n        # handoff from the CV worker for a one-item batch. Forcing the\n        # plain sequential path (no threads, no queues) avoids the failure\n        # mode entirely -- confirmed against real hung runs, not a guess.\n        self._ocr = PaddleOCRVL(use_queues=False)\n\n    @staticmethod\n    def _markdown_text(res) -> str:\n        md = getattr(res, "markdown", None)\n        if md is None and hasattr(res, "get"):\n            md = res.get("markdown")\n        if md is None:\n            return ""\n        texts = md.get("markdown_texts") if hasattr(md, "get") else None\n        if texts is None:\n            return str(md)\n        if isinstance(texts, (list, tuple)):\n            return "\\n".join(str(t) for t in texts)\n        return str(texts)\n\n    def _extract_printed_fields(self, image_path: str) -> dict:\n        self._ensure_loaded()\n        results = self._ocr.predict(str(image_path))\n        texts = [self._markdown_text(res) for res in results]\n        joined = "\\n".join(texts)\n\n        fields = {}\n        m = _INVOICE_NUMBER_RE.search(joined)\n        if m:\n            fields["invoice_number"] = m.group(1)\n        m = _BENEFICIARY_RE.search(joined)\n        if m:\n            fields["beneficiary"] = m.group(1).strip()\n        m = _AMOUNT_RE.search(joined)\n        if m:\n            try:\n                fields["amount"] = float(m.group(1).replace(",", ""))\n            except ValueError:\n                pass\n        m = _BANK_ACCOUNT_RE.search(joined)\n        if m:\n            fields["bank_account"] = m.group(1)\n        return fields\n\n    @staticmethod\n    def _decode_qr(image_path: str) -> dict | None:\n        import cv2\n\n        img = cv2.imread(str(image_path))\n        if img is None:\n            return None\n        detector = cv2.QRCodeDetector()\n        data, _points, _straight = detector.detectAndDecode(img)\n        if not data:\n            return None\n        try:\n            return json.loads(data)\n        except (json.JSONDecodeError, TypeError):\n            return None\n\n    def _compare_fields(self, printed: dict, qr: dict) -> list:\n        """Shared by score() and score_with_evidence() -- one comparison\n        pass, returns a list of (field, printed_val, qr_val, matched) so\n        the evidence-viewer wiring (Task #32) can show WHICH field tripped\n        the score, not just the number."""\n        results = []\n        for key in ("invoice_number", "beneficiary", "amount", "bank_account"):\n            if key not in printed:\n                continue\n            if key == "amount":\n                qr_val = qr.get("amount")\n                match = qr_val is not None and abs(float(printed[key]) - float(qr_val)) < 0.01\n            else:\n                qr_val = qr.get(key, "")\n                match = str(printed[key]).strip().lower() == str(qr_val).strip().lower()\n            results.append((key, printed[key], qr.get(key), match))\n        return results\n\n    def score(self, image_path: str | Path) -> float:\n        """Returns a tamper score in [0, 1]: the fraction of comparable\n        fields (of the ones OCR actually found) that mismatch between the\n        printed text and the QR-decoded payload, or 1.0 outright if the QR\n        can\'t be decoded at all. Returns 0.5 (genuinely unknown) if OCR\n        found none of the four fields."""\n        score, _evidence = self.score_with_evidence(image_path)\n        return score\n\n    def score_with_evidence(self, image_path: str | Path) -> tuple:\n        """Same score as score(), plus a human-readable evidence list\n        (which fields mismatched and how) -- added 2026-08-30 for Task #32\'s\n        evidence-viewer wiring. Still takes only a file path (Principle 13\n        unaffected) -- \'evidence\' here means the detector\'s own reasoning\n        trace, never ground truth."""\n        printed = self._extract_printed_fields(image_path)\n        qr = self._decode_qr(image_path)\n        if qr is None:\n            return 1.0, ["QR payload could not be decoded at all"]\n\n        comparisons = self._compare_fields(printed, qr)\n        if not comparisons:\n            return 0.5, ["OCR found none of the 4 comparable fields -- score is genuinely unknown"]\n\n        evidence = []\n        mismatched = 0\n        for key, printed_val, qr_val, matched in comparisons:\n            if matched:\n                evidence.append(f"{key}: matches QR payload")\n            else:\n                mismatched += 1\n                evidence.append(f"{key} mismatch: printed=\'{printed_val}\' vs QR=\'{qr_val}\'")\n        return mismatched / len(comparisons), evidence\n\n    def score_batch(self, image_paths: list) -> np.ndarray:\n        return np.array([self.score(p) for p in image_paths], dtype="float64")\n'

for _rel, _content in _FILES.items():
    _p = pathlib.Path('/content') / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_content, encoding='utf-8')

print(f'Wrote {len(_FILES)} files under /content/backend/')

## 3. Upload your data package

Upload the `document_fraud_data.zip` you created locally (see the instructions in cell 0).

In [ ]:
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print(f"Received {zip_name} ({len(uploaded[zip_name]):,} bytes)")

extract_dir = pathlib.Path("/content/_upload")
extract_dir.mkdir(parents=True, exist_ok=True)

# Windows' Compress-Archive sometimes writes entry names with literal
# backslashes instead of the zip-spec-required forward slashes (a known
# PowerShell/.NET quirk) -- Python's zipfile does not treat backslash as a
# path separator on ANY platform, so a plain extractall() here silently
# produces flat files with backslash characters baked into the filename
# instead of real subdirectories (confirmed against a real run: every
# document_attacks/train and document_attacks/held_out file landed as one
# flat file literally named "held_out\<case>.json" etc). Normalize
# separators ourselves before extracting so real nested folders come out.
with zipfile.ZipFile(zip_name) as zf:
    for info in zf.infolist():
        normalized = info.filename.replace("\\", "/")
        dest = extract_dir / normalized
        if info.is_dir() or normalized.endswith("/"):
            dest.mkdir(parents=True, exist_ok=True)
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info) as src, open(dest, "wb") as out:
            out.write(src.read())

target = pathlib.Path("/content/data/generated")
target.mkdir(parents=True, exist_ok=True)

def _place(name):
    src_matches = [p for p in extract_dir.rglob(name) if p.is_dir()]
    if not src_matches:
        raise FileNotFoundError(
            f"Could not find a '{name}' folder anywhere inside the uploaded zip -- "
            f"re-check the Compress-Archive command in cell 0's instructions."
        )
    dest = target / name
    if dest.exists():
        import shutil
        shutil.rmtree(dest)
    src_matches[0].rename(dest)
    print(f"Placed {name} -> {dest}")

_place("document_bonafide")
_place("document_attacks")

n_bonafide = len(list((target / "document_bonafide").glob("*.png")))
n_cases = len(list((target / "document_attacks").glob("*/*.json")))
print(f"\n{n_bonafide} bonafide PNGs, {n_cases} attack case JSON files ready.")
assert n_bonafide > 0, "No bonafide PNGs found after extraction -- check the zip contents."
assert n_cases > 0, "No attack case JSON files found after extraction -- check the zip contents."


## 4. Run the real evidence-gate script

Unmodified -- same script, same logic as the local repo, just run here because local GPU inference for this detector is unreliable on this machine.

In [ ]:
import subprocess, sys

proc = subprocess.run(
    [sys.executable, "evaluation/eval_document_consistency.py"],
    cwd="/content/backend", capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:", file=sys.stderr)
    print(proc.stderr, file=sys.stderr)
    raise SystemExit(f"eval_document_consistency.py exited with code {proc.returncode}")


## 5. Print the recorded result

Copy this JSON back into the chat -- same pattern as the GNN metrics snippet -- and it'll get merged into the real `backend/defend/models/metrics.json` on your machine.

In [ ]:
import json

metrics_path = "/content/backend/defend/models/metrics.json"
data = json.loads(open(metrics_path).read())
print(json.dumps({"document_consistency_detector": data["document_consistency_detector"]}, indent=2))
